# Problema del reparto de pizzas

## Encontrarás una descripción general del problema y luego tres apartados a resolver:
- Parte 1: utilización solo de bibicleta.
- Parte 2: utilización de moto eléctrica.
- Parte 3: cálculo de ganancia.

## Se pide:
- Hacer un programa ASP en clingo que resuelva cada uno de los apartados.
- Antes del programa de cada apartado, hay que explicar el diseño del programa, haciendo hincapié en la semántica de las reglas.


## 1. Carga y Configuración del Entorno Python

In [6]:
import clingo

def resolver_asp(titulo, codigo_asp, opciones=["0"]):
    """
    Función de soporte para instanciar, ejecutar y procesar modelos estables en Clingo.
    - titulos: Cadena de texto para etiquetar y organizar visualmente los resultados
    - codigo_asp: Cadena de texto con el código en ASP escrito en la sintaxis formal de Clingo/Gringo
    - opciones: Lista de parámetros de Clingo (ej. ["0"] indica a Clingo que calcule y devuelva todos los modelos)
    """
    print("=" * 74)
    print(f"  {titulo.upper()}")
    print("=" * 74)

    errores = []

    # Captura los mensajes generados por Clingo
    def logger(codigo, mensaje):
        errores.append(mensaje)

    ctl = clingo.Control(opciones, logger=logger)
    try:
        ctl.add("base", [], codigo_asp)
        ctl.ground([("base", [])])
    except RuntimeError:
        print("\n❌ ERROR EN EL PROGRAMA ASP\n")
        for error in errores:
            print(error)
        return

    modelos = []
    with ctl.solve(yield_=True) as handle:
        for modelo in handle:
            modelos.append(modelo.symbols(shown=True))

    if len(modelos) == 0:
        print("  [ UNSATISFIABLE ] No existe ninguna solución que cumpla las restricciones.\n")
        return

    print(f"\nNúmero de modelos: {len(modelos)}\n")
    for i, modelo in enumerate(modelos, 1):
        print(f"Modelo {i}:")
        print(modelo)

# **Descripción general del problema**

<div style="font-size: 16px">

La compañía **YaVoyPizza** reparte pizzas a domicilio en el área urbana de una ciudad. La compañía suministra los siguientes tipos de productos:

-	Pizzas medianas
-	Pizzas grandes
-	Botellas de refrescos de 1.5 l 

**YaVoyPizza** está comprometida con la sostenibilidad ambiental y solo reparte en bicicleta. Las capacidades de transporte de una bicicleta son:

-	Bicicleta
  
       * 2 pizzas grandes y 2 botellas de refresco, o

	   * 3 pizzas medianas y 2 botellas de refresco


Los pedidos pueden consistir en cualquier combinación de productos, pero cada pedido debe ser atendido con una única bicicleta y no puede superar los límites de capacidad indicados en el enunciado. Si un pedido supera los límites de capacidad que puede transportar una bicicleta, la empresa comunica al cliente que no se puede servir su pedido. Además, con el fin de que las pizzas no se enfríen, la empresa ha establecido que una bicicleta solo puede atender un pedido en cada viaje antes de volver a la base.

YaVoyPizza dispone de **2 bicicletas**. 


# **Parte 1**

<div style="font-size: 16px">
    
Dado un conjunto de pedidos registrados, determinar el subconjunto de pedidos que puede atenderse simultáneamente en una ronda de reparto (sin volver a base), asignando cada pedido a una bicicleta y respetando las restricciones indicadas en el enunciado.

Mostrar todos los modelos (combinaciones de subconjuntos de pedidos) que la compañía **YaVoyPizza** podría atender simultáneamente.

Cada equipo de trabajo definirá su conjunto de pedidos registrados.


In [7]:
part1 = '''

% Hechos (bicis disponibles):
bicicleta(b1; b2).


% Conjunto de pedidos con la estructura pedido(ID, Medianas, Grandes, Refrescos)

pedido(p1, 0, 2, 2).
pedido(p2, 3, 0, 1).
pedido(p3, 1, 1, 1).
pedido(p4, 4, 0, 0).
pedido(p5, 0, 0, 3).
pedido(p6, 4, 0, 2).
pedido(p7, 2, 1, 3).
pedido(p8, 0, 3, 2).
pedido(p9, 5, 0, 1).
pedido(p10, 1, 2, 4).
pedido(p11, 3, 1, 2).
pedido(p12, 0, 4, 1).
pedido(p13, 2, 2, 2).
pedido(p14, 3, 2, 3).
pedido(p15, 1, 0, 5).


% Regla generadora: Cada pedido se asigna a como máximo 1 bicicleta (o a ninguna)
0 { atender(P, B) : bicicleta(B) } 1 :- pedido(P, _, _, _).


% Restricciones

% Una bicicleta solo puede atender como máximo un pedido en esta ronda
:- bicicleta(B), #count { P : atender(P, B) } > 1.

% No se pueden superar los 2 refrescos por pedido
:- atender(P, B), pedido(P, _, _, R), R > 2.  

% No se puede superar el límite de volumen de pizzas (3*G + 2*M <= 6)
:- atender(P, B), pedido(P, M, G, _), 3*G + 2*M > 6.


% Predicados auxiliares
atendido(P) :- atender(P, _).
no_atendido(P) :- pedido(P, _, _, _), not atendido(P).


%  Resultados finales
#show atender/2.


'''

resolver_asp('Pizzas problema', part1)

  PIZZAS PROBLEMA

Número de modelos: 13

Modelo 1:
[]
Modelo 2:
[atender(p3,b2)]
Modelo 3:
[atender(p2,b1)]
Modelo 4:
[atender(p2,b1), atender(p3,b2)]
Modelo 5:
[atender(p3,b1)]
Modelo 6:
[atender(p2,b2)]
Modelo 7:
[atender(p3,b1), atender(p2,b2)]
Modelo 8:
[atender(p1,b2)]
Modelo 9:
[atender(p2,b1), atender(p1,b2)]
Modelo 10:
[atender(p3,b1), atender(p1,b2)]
Modelo 11:
[atender(p1,b1)]
Modelo 12:
[atender(p1,b1), atender(p3,b2)]
Modelo 13:
[atender(p1,b1), atender(p2,b2)]


# **Parte 2**

<div style="font-size: 16px">

    
La compañía **YaVoyPizza** ha decidido incorporar **1 moto eléctrica** como vehículo para el reparto de pedidos. La capacidad de transporte de la moto es:

-	Moto
  
       * 3 pizzas grandes y 4 botellas de refresco, o

       * 5 pizzas medianas y 5 botellas de refresco, o

       * 4 pizzas grandes, 2 pizzas medianas y 4 botellas de refresco

Ampliar y/o modificar el programa de la Parte 1 para incorporar este nuevo vehículo y sus restricciones. El resto de las especificaciones y condiciones del problema no cambian. Esto es, cada pedido se transporta en un único vehículo (bicicleta o moto), y un vehículo lleva solo el pedido para un cliente.

Mostrar los subconjuntos de pedidos que **YaVoyPizza** podría atender simultáneamente en una ronda de reparto (sin considerar que los vehículos vuelven a base), asignando cada pedido a un único vehículo y respetando las restricciones de disponibilidad de vehículos y sus capacidades.


In [8]:
part2 = '''

% Hechos (bicis disponibles):
bicicleta(b1; b2).
moto(m1).


% Unificación de vehículos
vehiculo(V) :- bicicleta(V).
vehiculo(V) :- moto(V).

% Conjunto de pedidos con la estructura pedido(ID, Medianas, Grandes, Refrescos)

pedido(p1, 0, 2, 2).
pedido(p2, 3, 0, 1).
pedido(p3, 1, 1, 1).
pedido(p4, 4, 0, 0).
pedido(p5, 0, 0, 3).
pedido(p6, 4, 0, 2).
pedido(p7, 2, 1, 3).
pedido(p8, 0, 3, 2).
pedido(p9, 5, 0, 1).
pedido(p10, 1, 2, 4).
pedido(p11, 3, 1, 2).
pedido(p12, 0, 4, 1).
pedido(p13, 2, 2, 2).
pedido(p14, 3, 2, 3).
pedido(p15, 1, 0, 5).

% Regla generadora
% Cada pedido se asigna como máximo a un unico vehiculo
0 { atender(P, V) : vehiculo(V) } 1 :- pedido(P, _, _, _).


% Restricciones generales
% Un vehículo solo puede llevar como máximo 1 pedido en cada ronda
:- vehiculo(V), #count { P : atender(P, V) } > 1.


% Restricciones de bici
%  Máximo 2 refrescos por bicicleta
:- atender(P, B), bicicleta(B), pedido(P, _, _, R), R > 2.

%  Máximo volumen de pizzas en bicicleta (3*G + 2*M <= 6)
:- atender(P, B), bicicleta(B), pedido(P, M, G, _), 3*G + 2*M > 6.


% Restricciones de moto
% Hasta 3 grandes y 4 refrescos
valido_moto(P) :- pedido(P, M, G, R), G <= 3, M == 0, R <= 4.

% Hasta 5 medianas y 5 refrescos
valido_moto(P) :- pedido(P, M, G, R), G == 0, M <= 5, R <= 5.

% Hasta 4 grandes, 2 medianas y 4 refrescos
valido_moto(P) :- pedido(P, M, G, R), G <= 4, M <= 2, R <= 4.

% Prohibido asignar a la moto un pedido que no cumpla ninguno de los 3 casos
:- atender(P, M), moto(M), not valido_moto(P).


% Predicciones auxiliares
atendido(P) :- atender(P, _).
no_atendido(P) :- pedido(P, _, _, _), not atendido(P).


% Resultados
#show atender/2.
'''

resolver_asp('Pizzas problema', part2)

  PIZZAS PROBLEMA

Número de modelos: 164

Modelo 1:
[]
Modelo 2:
[atender(p1,m1)]
Modelo 3:
[atender(p3,b1)]
Modelo 4:
[atender(p3,b1), atender(p1,m1)]
Modelo 5:
[atender(p2,b2)]
Modelo 6:
[atender(p2,b2), atender(p1,m1)]
Modelo 7:
[atender(p3,b1), atender(p2,b2)]
Modelo 8:
[atender(p3,b1), atender(p2,b2), atender(p1,m1)]
Modelo 9:
[atender(p1,b2)]
Modelo 10:
[atender(p3,b1), atender(p1,b2)]
Modelo 11:
[atender(p1,b1)]
Modelo 12:
[atender(p1,b1), atender(p2,b2)]
Modelo 13:
[atender(p3,b2)]
Modelo 14:
[atender(p3,b2), atender(p1,m1)]
Modelo 15:
[atender(p1,b1), atender(p3,b2)]
Modelo 16:
[atender(p2,b1)]
Modelo 17:
[atender(p2,b1), atender(p1,m1)]
Modelo 18:
[atender(p2,b1), atender(p1,b2)]
Modelo 19:
[atender(p2,b1), atender(p3,b2)]
Modelo 20:
[atender(p2,b1), atender(p3,b2), atender(p1,m1)]
Modelo 21:
[atender(p2,m1)]
Modelo 22:
[atender(p3,b2), atender(p2,m1)]
Modelo 23:
[atender(p3,b1), atender(p2,m1)]
Modelo 24:
[atender(p1,b1), atender(p2,m1)]
Modelo 25:
[atender(p1,b1), atender(

# **Parte 3**

<div style="font-size: 16px">

    
La compañía **YaVoyPizza** desea saber cuál es el subconjunto de pedidos que puede atenderse simultáneamente que le reportaría mayor beneficio, sabiendo que:

-	El precio de una pizza grande es 12€.
-	El precio de una pizza mediana es 10€.
-	El precio de un refresco es 4€.

Mostrar las combinaciones de pedidos que podría atenderse simultáneamente en una ronda de reparto (sin considerar que los vehículos vuelven a base), y lo que ganaría **YaVoyPizza** en cada ronda.

In [10]:
part3 = '''


% Hechos (bicis disponibles):
bicicleta(b1; b2).
moto(m1).


% Unificación de vehículos
vehiculo(V) :- bicicleta(V).
vehiculo(V) :- moto(V).

% Conjunto de pedidos con la estructura pedido(ID, Medianas, Grandes, Refrescos)

pedido(p1, 0, 2, 2).
pedido(p2, 3, 0, 1).
pedido(p3, 1, 1, 1).
pedido(p4, 4, 0, 0).
pedido(p5, 0, 0, 3).
pedido(p6, 4, 0, 2).
pedido(p7, 2, 1, 3).
pedido(p8, 0, 3, 2).
pedido(p9, 5, 0, 1).
pedido(p10, 1, 2, 4).
pedido(p11, 3, 1, 2).
pedido(p12, 0, 4, 1).
pedido(p13, 2, 2, 2).
pedido(p14, 3, 2, 3).
pedido(p15, 1, 0, 5).

% Regla generadora
% Cada pedido se asigna como máximo a un unico vehiculo
0 { atender(P, V) : vehiculo(V) } 1 :- pedido(P, _, _, _).


% Restricciones generales
% Un vehículo solo puede llevar como máximo 1 pedido en cada ronda
:- vehiculo(V), #count { P : atender(P, V) } > 1.


% Restricciones de bici
%  Máximo 2 refrescos por bicicleta
:- atender(P, B), bicicleta(B), pedido(P, _, _, R), R > 2.

%  Máximo volumen de pizzas en bicicleta (3*G + 2*M <= 6)
:- atender(P, B), bicicleta(B), pedido(P, M, G, _), 3*G + 2*M > 6.


% Restricciones de moto
% Hasta 3 grandes y 4 refrescos
valido_moto(P) :- pedido(P, M, G, R), G <= 3, M == 0, R <= 4.

% Hasta 5 medianas y 5 refrescos
valido_moto(P) :- pedido(P, M, G, R), G == 0, M <= 5, R <= 5.

% Hasta 4 grandes, 2 medianas y 4 refrescos
valido_moto(P) :- pedido(P, M, G, R), G <= 4, M <= 2, R <= 4.

% Prohibido asignar a la moto un pedido que no cumpla ninguno de los 3 casos
:- atender(P, M), moto(M), not valido_moto(P).



% Predicciones auxiliares
atendido(P) :- atender(P, _).
no_atendido(P) :- pedido(P, _, _, _), not atendido(P).


% Calculo de beneficios
benificio_pedido(P, 10 * M, 12 * G, 4 * R) :- pedido(P, M, G, R).
beneficio_total(Total) :- Total = #sum { B,P : atender(P,_), beneficio_pedido(P,B) }.


% Optimizacion
#maximize { B,P : atender(P,_), beneficio_pedido(P,B) }.

% Resultados
#show atender/2.
#show beneficio_total/1.

'''

resolver_asp('Pizzas problema', part3)



  PIZZAS PROBLEMA

Número de modelos: 164

Modelo 1:
[beneficio_total(0)]
Modelo 2:
[atender(p1,m1), beneficio_total(0)]
Modelo 3:
[atender(p3,b1), beneficio_total(0)]
Modelo 4:
[atender(p3,b1), atender(p1,m1), beneficio_total(0)]
Modelo 5:
[atender(p1,b1), beneficio_total(0)]
Modelo 6:
[atender(p2,b2), beneficio_total(0)]
Modelo 7:
[atender(p2,b2), atender(p1,m1), beneficio_total(0)]
Modelo 8:
[atender(p3,b1), atender(p2,b2), beneficio_total(0)]
Modelo 9:
[atender(p3,b1), atender(p2,b2), atender(p1,m1), beneficio_total(0)]
Modelo 10:
[atender(p1,b1), atender(p2,b2), beneficio_total(0)]
Modelo 11:
[atender(p1,b2), beneficio_total(0)]
Modelo 12:
[atender(p3,b1), atender(p1,b2), beneficio_total(0)]
Modelo 13:
[atender(p3,b2), beneficio_total(0)]
Modelo 14:
[atender(p3,b2), atender(p1,m1), beneficio_total(0)]
Modelo 15:
[atender(p1,b1), atender(p3,b2), beneficio_total(0)]
Modelo 16:
[atender(p2,b1), beneficio_total(0)]
Modelo 17:
[atender(p2,b1), atender(p1,m1), beneficio_total(0)]
Modelo